# NB-02_controles_y_ajustes_iniciales

Process Flow SAS: **Controles y ajustes iniciales** — `PFD-I32F27oZ8ICuby6y`

In [ ]:
# ========= Parámetros =========
# Variables macro del SAS original. El .egp NO las define (venían del
# entorno SAS): su valor sale de la entrevista B4 o de
# project_config.yaml → run.macro_params, o se inyecta acá
# (celda 'parameters' de papermill).

ANIO = None  # &ANIO — nadie declaró su valor
TRIM = None  # &TRIM — nadie declaró su valor
anio = None  # &anio — nadie declaró su valor

faltantes = [n for n, v in {"ANIO": ANIO, "TRIM": TRIM, "anio": anio}.items() if v is None]
if faltantes:
    raise ValueError(f"Parámetros sin valor: {faltantes}")

In [ ]:
# ========= Celda 1: Configuración =========
import pandas as pd
import numpy as np
import os
import sqlalchemy
import datetime
from pathlib import Path
import requests
from sqlalchemy import text
import bcchapi

# Conexión a BD — editable acá; SASMIG_DB_URL (orquestador) tiene
# prioridad si está definida (SUPUESTO: verificar servidor y base
# antes de correr contra datos reales).
config_db = (
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "SERVER=PLATDAT,1433;"
    "DATABASE=GOBGENER;"
    "Authentication=ActiveDirectoryIntegrated;"
    "Encrypt=yes;"
    "TrustServerCertificate=no;"
    "MARS_Connection=Yes;"
)
engine = sqlalchemy.create_engine(
    os.environ.get("SASMIG_DB_URL", f"mssql+pyodbc:///?odbc_connect={config_db}"),
    pool_pre_ping=True,
    fast_executemany=True,
)
# Sesión de BD del notebook — espejo de la sesión WORK de SAS: las
# tablas temporales #tmp viven en ESTA conexión y mueren al cerrar el
# kernel. AUTOCOMMIT: cada statement commitea, como los pasos de SAS.
work_conn = engine.connect().execution_options(isolation_level="AUTOCOMMIT")

# Logging liviano de resultados — aprobado en la entrevista (Fase 4)
_LOG_PATH = Path("log") / "NB-02_controles_y_ajustes_iniciales.log"
_LOG_PATH.parent.mkdir(parents=True, exist_ok=True)
def _log(label, value=None):
    """Una línea por celda: imprime y persiste. Jamás rompe la corrida."""
    try:
        if hasattr(value, "shape"):
            detail = f"{value.shape[0]} filas x {value.shape[1]} cols"
        elif isinstance(value, int):
            detail = f"{value} filas"
        elif value is None:
            detail = "ok"
        else:
            detail = str(value)
        line = f"[{datetime.datetime.now():%Y-%m-%d %H:%M:%S}] {label}: {detail}"
        print(line)
        with open(_LOG_PATH, "a", encoding="utf-8") as fh:
            fh.write(line + "\n")
    except Exception:
        pass  # el log nunca puede tumbar el notebook
with open(_LOG_PATH, "a", encoding="utf-8") as _fh:
    _fh.write(f"\n=== corrida {datetime.datetime.now():%Y-%m-%d %H:%M:%S} ===\n")


## S1

Construye y actualiza las bases de ajuste (SIFMI, dividendos hogares, PIB vía API BCCh, utilidades reinvertidas, T_CCAST, ajustes varios, FBCF sector financiero) usadas para conciliar las cuentas nacionales trimestrales del sector financiero

*confianza: low · verificador: revise · SAS: PROC IMPORT + PROC SQL UNION ALL/CREATE TABLE/UPDATE/DELETE + PROC APPEND + PROC HTTP (API BDE) + DATA step concatenación*

In [ ]:
# ========= S1 =========
# IMPORTA DATA - SIFMI desde Excel (ruta relativa; M-001 elimina hardcode /samba)
sifmi_path = Path("datos") / "CONTROLES" / "SIFMI.xlsx"
sifmi = pd.read_excel(sifmi_path, sheet_name="SIFMI_SAS")
_log("sifmi", sifmi)


In [ ]:
# CREA BASE DE DATOS SIFMI. CIERRE 2021: INCORPORA SECTORES GOB, SEGUROS Y AUXILIARES
fecha_hoy = pd.Timestamp.today().normalize()
filas_sifmi = []
for sector, col, c_cagente, c_entrada, signo in [
    (51, "Empresas_pagados", "321", "D", -1),
    (511, "Hogares_pagados", "321", "D", -1),
    (41, "Gob_pagados", "321", "D", -1),
    (35, "Seg_pagados", "321", "D", -1),
    (36, "Aux_pagados", "321", "D", -1),
]:
    filas_sifmi.append(pd.DataFrame({
        "MONEDA": "P", "AÑO": sifmi["Año"], "TRIM": sifmi["Trimestre"], "SECTOR": sector,
        "C_CUENTA": "YG", "C_CAGENTE": c_cagente, "C_ENTRADA": c_entrada,
        "DATO": sifmi[col] * signo, "C_SCN": "D.41", "N_SCN": "Intereses",
        "FUENTE": "DI_Aj_SIFMI", "FECHA": fecha_hoy,
    }))
# fila agregada: suma de los 5 sectores pagados, con signo de deuda (H) sobre C_CAGENTE 53
filas_sifmi.append(pd.DataFrame({
    "MONEDA": "P", "AÑO": sifmi["Año"], "TRIM": sifmi["Trimestre"], "SECTOR": 321,
    "C_CUENTA": "YG", "C_CAGENTE": "53", "C_ENTRADA": "H",
    "DATO": (sifmi["Hogares_pagados"] + sifmi["Empresas_pagados"] + sifmi["Gob_pagados"] + sifmi["Seg_pagados"] + sifmi["Aux_pagados"]) * -1,
    "C_SCN": "D.41", "N_SCN": "Intereses", "FUENTE": "DI_Aj_SIFMI", "FECHA": fecha_hoy,
}))
for sector, col, c_cagente, c_entrada in [
    (51, "Empresas_recibidos", "321", "H"),
    (511, "Hogares_recibidos", "321", "H"),
    (41, "Gob_recibidos", "321", "H"),
    (35, "Seg_recibidos", "321", "H"),
    (36, "Aux_recibidos", "321", "H"),
]:
    filas_sifmi.append(pd.DataFrame({
        "MONEDA": "P", "AÑO": sifmi["Año"], "TRIM": sifmi["Trimestre"], "SECTOR": sector,
        "C_CUENTA": "YG", "C_CAGENTE": c_cagente, "C_ENTRADA": c_entrada,
        "DATO": sifmi[col], "C_SCN": "D.41", "N_SCN": "Intereses",
        "FUENTE": "DI_Aj_SIFMI", "FECHA": fecha_hoy,
    }))
filas_sifmi.append(pd.DataFrame({
    "MONEDA": "P", "AÑO": sifmi["Año"], "TRIM": sifmi["Trimestre"], "SECTOR": 321,
    "C_CUENTA": "YG", "C_CAGENTE": "53", "C_ENTRADA": "D",
    "DATO": (sifmi["Hogares_recibidos"] + sifmi["Empresas_recibidos"] + sifmi["Gob_recibidos"] + sifmi["Seg_recibidos"] + sifmi["Aux_recibidos"]),
    "C_SCN": "D.41", "N_SCN": "Intereses", "FUENTE": "DI_Aj_SIFMI", "FECHA": fecha_hoy,
}))
sifmi_final = pd.concat(filas_sifmi, ignore_index=True)
# DELETE FROM TABLAS.SIFMI WHERE AÑO=.  -> AÑO nulo se descarta antes del reemplazo
sifmi_final = sifmi_final[sifmi_final["AÑO"].notna()]
with engine.begin() as conn:
    res = conn.execute(text("DELETE FROM TABLAS.dbo.SIFMI"))
    _log("DELETE TABLAS.dbo.SIFMI", res.rowcount)
sifmi_final.to_sql("SIFMI", engine, schema="dbo", if_exists="append", index=False)
_log("sifmi_final", sifmi_final)


In [ ]:
# CALCULA PROMEDIO DEL TRIMESTRE A TRABAJAR (AJUSTE INICIO PERIODO COYUNTURA)
sql_rp_hh_sum = """
SELECT MONEDA, AÑO, TRIM, SECTOR, C_CUENTA, C_CAGENTE, C_ENTRADA, C_SCN, N_SCN, FUENTE, SUM(DATO) AS DATO
FROM TABLAS.dbo.RP_HH
WHERE AÑO >= 2008 AND TRIM = :trim
GROUP BY MONEDA, AÑO, TRIM, SECTOR, C_CUENTA, C_CAGENTE, C_ENTRADA, C_SCN, N_SCN, FUENTE
"""
rp_hh_sum = pd.read_sql(text(sql_rp_hh_sum), engine, params={"trim": TRIM})
# PARA ELIMINAR PERIODO DE COYUNTURA EN CASO QUE SE CORRA ESTE PROG VARIAS VECES
rp_hh_sum = rp_hh_sum[~((rp_hh_sum["AÑO"] == ANIO) & (rp_hh_sum["TRIM"] == TRIM))]
_log("rp_hh_sum", rp_hh_sum)


In [ ]:
# Promedio de coyuntura (MEAN) con AÑO fijado a ANIO
rp_hh_av = (
    rp_hh_sum.groupby(["MONEDA", "TRIM", "SECTOR", "C_CUENTA", "C_CAGENTE", "C_ENTRADA", "C_SCN", "N_SCN", "FUENTE"], as_index=False)["DATO"]
    .mean()
)
rp_hh_av["AÑO"] = ANIO
rp_hh_av["FECHA"] = pd.Timestamp.today().normalize()
rp_hh_av["PROC"] = "P"
_log("rp_hh_av", rp_hh_av)


In [ ]:
# ELIMINA DATOS DE COYUNTURA EN TABLA PRINCIPAL (anti-join sobre AÑO/TRIM/PROC)
rp_hh_av_keys = rp_hh_av[["AÑO", "TRIM", "PROC"]].drop_duplicates()
work_conn.execute(text("DROP TABLE IF EXISTS #rp_hh_av_keys"))
rp_hh_av_keys.to_sql("#rp_hh_av_keys", work_conn, if_exists="replace", index=False)
sql_delete_rp_hh = """
DELETE t FROM TABLAS.dbo.RP_HH t
WHERE EXISTS (
    SELECT 1 FROM #rp_hh_av_keys k
    WHERE k.AÑO = t.AÑO AND k.TRIM = t.TRIM AND k.PROC = t.PROC
)
"""
res = work_conn.execute(text(sql_delete_rp_hh))
_log("DELETE TABLAS.dbo.RP_HH (coyuntura)", res.rowcount)


In [ ]:
# ANEXA PROMEDIO DIVIDENDOS A BASE RP_HH (PROC APPEND force -> append tal cual)
cols_rp_hh = ["MONEDA", "AÑO", "TRIM", "SECTOR", "C_CUENTA", "C_CAGENTE", "C_ENTRADA", "C_SCN", "N_SCN", "FUENTE", "DATO"]
rp_hh_av[cols_rp_hh].to_sql("RP_HH", engine, schema="dbo", if_exists="append", index=False)
# nota: FORCE de PROC APPEND descarta columnas de apoyo (FECHA, PROC) que RP_HH no tiene
_log("append RP_HH", len(rp_hh_av))


In [ ]:
# PIB a precios corrientes - API BDE (host si3.bcentral.cl declarado mode=sdk -> bcchapi)
siete = bcchapi.Siete(os.environ["BDE_USER"], os.environ["BDE_PASS"])
serie_pib = siete.cuadro(
    series=["F032.PIB.FLU.N.CLP.EP18.Z.Z.0.T"],
    nombres=["valor"],
)


In [ ]:
# Normaliza fecha/valor de la serie PIB y arma TABLAS.PIB
pib = serie_pib.reset_index().rename(columns={"index": "fecha"})
pib["AÑO"] = pd.to_datetime(pib["fecha"]).dt.year
pib["TRIM"] = pd.to_datetime(pib["fecha"]).dt.month
pib["DATO"] = pd.to_numeric(pib["valor"], errors="coerce") * 1000
pib["FECHA"] = pd.Timestamp.today().normalize()
pib = pib[pib["AÑO"].notna()][["AÑO", "TRIM", "DATO", "FECHA"]]
# UPDATE TRIM: 4->2, 7->3, 10->4 (meses de reporte trimestral -> número de trimestre)
pib["TRIM"] = pib["TRIM"].replace({4: 2, 7: 3, 10: 4})
with engine.begin() as conn:
    res = conn.execute(text("DELETE FROM TABLAS.dbo.PIB"))
    _log("DELETE TABLAS.dbo.PIB", res.rowcount)
pib.to_sql("PIB", engine, schema="dbo", if_exists="append", index=False)
_log("pib", pib)


In [ ]:
raise NotImplementedError("Series CNT (serie_1..serie_9) requieren 9 llamadas a la API BDE (F033.IRM, IRMP, TCE, TCEP, AEX, FKF, VAX, XBS, IBS) con timeseries distintas y su UNION en TABLAS.CNT; el patron se repite igual que PIB pero no hay confirmacion del metodo bcchapi (cuadro multi-serie vs llamadas individuales) para resolverlo con confianza en esta pasada")


In [ ]:
# UTILIDADES REINVERTIDAS DEL SECTOR FINANCIERO - import desde Excel (ruta relativa, M-001)
ur_sf_path = Path("datos") / "INFO_AUX" / "UT_REINVERTIDAS_CR18.xlsx"
ur_sf_cr18 = pd.read_excel(ur_sf_path, sheet_name="UR_SF", skiprows=1)
# NO SE INCORPORA APERTURA PORQUE AFECTA MUCHO EL PTMO NETO DE LOS SEGUROS
ur_sf_cr18["SECTOR"] = ur_sf_cr18["SECTOR"].replace({35: 321})
with engine.begin() as conn:
    res = conn.execute(text("DELETE FROM TABLAS.dbo.UR_SF_CR18"))
    _log("DELETE TABLAS.dbo.UR_SF_CR18", res.rowcount)
ur_sf_cr18.to_sql("UR_SF_CR18", engine, schema="dbo", if_exists="append", index=False)
_log("ur_sf_cr18", ur_sf_cr18)


In [ ]:
# IMPORTA DATA DE CCAST y reemplaza TABLAS.T_CCAST
t_ccast_path = Path("datos") / "INFO_AUX" / "T_CCAST.xlsx"
t_ccast = pd.read_excel(t_ccast_path, sheet_name="T_CCAST")
with engine.begin() as conn:
    res = conn.execute(text("DELETE FROM TABLAS.dbo.T_CCAST"))
    _log("DELETE TABLAS.dbo.T_CCAST", res.rowcount)
t_ccast.to_sql("T_CCAST", engine, schema="dbo", if_exists="append", index=False)
_log("t_ccast", t_ccast)


In [ ]:
# ELIMINA DE AJUSTE BONOS AÑO DE COYUNTURA PARA RECALCULAR DE NUEVO POR CAMBIO DE CUENTAS INDIVIDUALES
with engine.begin() as conn:
    res = conn.execute(text("DELETE FROM TABLAS.dbo.AJUSTE_BONOS WHERE AÑO >= :anio"), {"anio": ANIO})
    _log("DELETE TABLAS.dbo.AJUSTE_BONOS (coyuntura)", res.rowcount)


In [ ]:
# IMPORTA AJUSTES VARIOS DEP Y ACCIONES
aj_varios_path = Path("datos") / "INFO_AUX" / "aj_cnsi.xlsx"
aj_varios = pd.read_excel(aj_varios_path, sheet_name="AJUSTES_VARIOS")


In [ ]:
# CIERRE 2021: IMPORTA AJUSTES TRANSFERENCIAS CORRIENTES DE EMPRESAS POR CDR18
aj_d443_cr18 = pd.read_excel(aj_varios_path, sheet_name="base_aj_d443_cr18")
aj_d443_cr18 = aj_d443_cr18[aj_d443_cr18["año"].notna()]


In [ ]:
# Concatena ajustes varios + AJ_D443_CR18 y reemplaza TABLAS.AJ_VARIOS (SET TABLAS.AJ_VARIOS AJ_D443_CR18)
aj_varios_full = pd.concat([aj_varios, aj_d443_cr18], ignore_index=True)
with engine.begin() as conn:
    res = conn.execute(text("DELETE FROM TABLAS.dbo.AJ_VARIOS"))
    _log("DELETE TABLAS.dbo.AJ_VARIOS", res.rowcount)
aj_varios_full.to_sql("AJ_VARIOS", engine, schema="dbo", if_exists="append", index=False)
_log("aj_varios_full", aj_varios_full)


In [ ]:
# CIERRE 2021: INCORPORA AJUSTE A FBCF SECTOR FINANCIERO (reemplaza TABLAS.FBCF_SF)
fbcf_sf_path = Path("datos") / "INFO_AUX" / "aj_fbcf_sf.xlsx"
fbcf_sf = pd.read_excel(fbcf_sf_path, sheet_name="BASE")
with engine.begin() as conn:
    res = conn.execute(text("DELETE FROM TABLAS.dbo.FBCF_SF"))
    _log("DELETE TABLAS.dbo.FBCF_SF", res.rowcount)
fbcf_sf.to_sql("FBCF_SF", engine, schema="dbo", if_exists="append", index=False)
_log("fbcf_sf", fbcf_sf)


In [ ]:
raise NotImplementedError("BONOS_RF_EXT, BONOS_RF_EXT_BI, RP_AUXFIN, AF32_51_6, AF32_36_6 leen desde archivos .sas7bdat en rutas /sasdata (BASE_DEUDA_EMV.sas7bdat, BD_CTSI_CIERRE.sas7bdat) que no estan en input_datasets, file_imports ni db_aliases del nodo: no son datasets WORK previos ni tablas de GOBGENER/TABLAS. No se puede resolver el origen del dato sin que el plan declare estos .sas7bdat como file_imports o tablas de BD")


## S1_d

Calcula transferencias de capital (D.9) desde gobierno general a empresas públicas por trimestre y las reemplaza en la tabla histórica para el año de coyuntura en curso

*confianza: medium · verificador: approve · SAS: PROC SQL CREATE TABLE con LEFT JOIN de 9 condiciones + GROUP BY + DELETE/APPEND idempotente por año*

In [ ]:
# ========= S1_d =========
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_cgr"))
# año debe ser mayor o igual a 2005 en cierre de año y el corriente en coyuntura
# año interpolado como entero (no :param) para que la #tmp sobreviva
sql_ejec_cgr = f"""
SELECT *
INTO #ejec_cgr
FROM GOBGENER.dbo.EJECUCION
WHERE AÑO >= {int(ANIO)}
"""
work_conn.execute(text(sql_ejec_cgr))


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #tk_ejec"))
# obtiene transferencias de capital a empresas para imputar en la síntesis
sql_tk_ejec = """
SELECT t1.AÑO,
       (CASE WHEN t1.MES IN (1,2,3) THEN 1
             WHEN t1.MES IN (4,5,6) THEN 2
             WHEN t1.MES IN (7,8,9) THEN 3
             ELSE 4 END) AS TRIM,
       5101 AS C_SI,
       'D.9' AS C_INSTRUMENTO_SCN,
       'Capital' AS C_CUENTA,
       'H' AS C_ENTRADA,
       SUM(t1.DEVENG / 1000000.0) AS Dato
INTO #tk_ejec
FROM #ejec_cgr t1
LEFT JOIN GOBGENER.dbo.CR18_T_SCN_2 t2
    ON t1.C_PARTIDA = t2.C_PARTIDA AND t1.C_CAPITULO = t2.C_CAPITULO AND t1.C_PROGRAMA = t2.C_PROGRAMA
   AND t1.C_ENTIDAD = t2.ENTIDAD AND t1.C_TIPO_CUENTA = t2.T_CUENTA AND t1.C_CUENTA = t2.C_CUENTA
   AND t1.C_ITEM = t2.C_ITEM AND t1.C_ASIGNACION = t2.C_ASIGNACION AND t1.C_ANALITICO = t2.C_ANALITICO
WHERE t1.C_CUENTA IN ('05','13','24','33')
  AND (t2.OBS IS NULL OR t2.OBS NOT IN ('CR18_difcoy','CR18_difact'))
  AND t1.C_ENTIDAD NOT IN (5601, 10201)
  AND (t2.C_SCN IS NULL OR t2.C_SCN NOT IN ('TC'))
  AND t1.MONEDA = 'P'
  AND t2.C_SCN IN ('D91','D92','D93','D99')
  AND t1.C_TIPO_CUENTA = 'G'
  AND t2.C_CAGENTE IN ('S11','S11_EPU','tkemppúb')
  AND t2.N_CUENTA = 'capital'
GROUP BY t1.AÑO,
         (CASE WHEN t1.MES IN (1,2,3) THEN 1
               WHEN t1.MES IN (4,5,6) THEN 2
               WHEN t1.MES IN (7,8,9) THEN 3
               ELSE 4 END)
"""
work_conn.execute(text(sql_tk_ejec))
tk_ejec = pd.read_sql(text("SELECT * FROM #tk_ejec"), work_conn)
_log("tk_ejec", tk_ejec)


In [ ]:
# ELIMINA DATOS DE AÑO DE COYUNTURA EN TABLA PRINCIPAL
with engine.begin() as conn:
    res = conn.execute(text("DELETE FROM TABLAS.dbo.T_TK_GG_EPU WHERE AÑO >= :anio"), {"anio": int(ANIO)})
    _log("DELETE TABLAS.dbo.T_TK_GG_EPU", res.rowcount)

# DELETE from TK_EJEC where AÑO<&ANIO (se aplica sobre la #tmp de sesión)
res = work_conn.execute(text("DELETE FROM #tk_ejec WHERE AÑO < :anio"), {"anio": int(ANIO)})
_log("DELETE #tk_ejec", res.rowcount)


In [ ]:
# ANEXA TK DE COYUNTURA A TABLA PRINCIPAL (PROC APPEND FORCE): server-side, sin pasar por pandas
cols_tk_gg_epu = "AÑO, TRIM, C_SI, C_INSTRUMENTO_SCN, C_CUENTA, C_ENTRADA, Dato"
sql_append_tk_gg_epu = f"""
INSERT INTO TABLAS.dbo.T_TK_GG_EPU ({cols_tk_gg_epu})
SELECT {cols_tk_gg_epu}
FROM #tk_ejec
"""
res = work_conn.execute(text(sql_append_tk_gg_epu))
_log("APPEND TABLAS.dbo.T_TK_GG_EPU", res.rowcount)


In [ ]:
# delete EJEC_CGR, TK_EJEC (limpieza de temporales de sesión)
for t in ["#ejec_cgr", "#tk_ejec"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


## S1_b

Calcula el porcentaje de depósitos bancarios (DPC/DPL) atribuible a hogares en fondos mutuos por sector/trimestre, imputando 2003-2005 con el valor de 2006 trim 1, y actualiza la serie histórica TABLAS.DEP_HH_FM desde el año de recálculo

*confianza: low · verificador: approve · SAS: Cadena PROC SQL/DATA step: joins, agregaciones, divisiones e imputación determinística, con UPDATE/DELETE/APPEND final a tabla de BD*

In [ ]:
# ========= S1_b =========
# CALCULA % DE DEPÓSITOS PARA HOGARES EN FFMM
# ID_FM: nace de un archivo SAS externo (.sas7bdat) no migrado a la BD del proyecto
raise NotImplementedError("ID_FM se construye desde IDENTIFICA_FFMM.sas7bdat (ruta SAS /sasdata/...), fuente no disponible en el catálogo de conexiones del proyecto; se requiere definir de dónde sale este dataset en el entorno Python")

In [ ]:
# 1. CALCULA % A HOGARES POR ROL DEL FONDO
# PROC IMPORT: BD_Patrimonio.xlsx, hoja base_datos, encabezados en fila 1, datos desde fila 2
ruta_patrimonio = Path("samba") / "BCCH" / "GEM_DCNI" / "02_CNSI" / "12_SI_FI" / "33901_FFMM" / "mensual" / "BD_Patrimonio.xlsx"
base_datos = pd.read_excel(ruta_patrimonio, sheet_name="base_datos", header=0)
_log("base_datos", base_datos)

In [ ]:
# UPDATE base_datos SET MES=12 WHERE AÑO<2017 AND MES=. (missing numérico -> NULL/NaN)
base_datos.loc[(base_datos["AÑO"] < 2017) & (base_datos["MES"].isna()), "MES"] = 12

In [ ]:
# DATO A HOGARES
base_datos["RUN_SDV"] = base_datos["RUN"].astype(str).str[:4]
hogar_mask = base_datos["DESTINO"].isin(["HOGARES", "EMPRE_HOGAR"])
base_hh = base_datos[hogar_mask].copy()
base_hh["PATRI_AJUST"] = np.where(base_hh["DESTINO"] == "EMPRE_HOGAR", base_hh["PATRI_T"] / 2, base_hh["PATRI_T"])
dato_hh = (
    base_hh.groupby(["AÑO", "MES", "RUN", "tipo_fondo"], as_index=False)
    .agg(DATO=("PATRI_AJUST", "sum"), RUN_SDV=("RUN_SDV", "first"))
)

# DATO TOTAL
dato_tot = (
    base_datos.groupby(["AÑO", "MES", "RUN"], as_index=False)
    .agg(DATO=("PATRI_T", "sum"))
)
dato_tot["RUN_SDV"] = dato_tot["RUN"].astype(str).str[:4]

# DATO_EST: proporción hogar/total por RUN
dato_est = dato_hh.merge(
    dato_tot, on=["AÑO", "MES", "RUN"], how="inner", suffixes=("_hh", "_tot")
)
dato_est["run_sdv"] = pd.to_numeric(dato_est["RUN_SDV_hh"], errors="coerce")
dato_est["EST"] = dato_est["DATO_hh"] / dato_est["DATO_tot"]
dato_est = dato_est[["AÑO", "MES", "RUN", "run_sdv", "tipo_fondo", "EST"]]
_log("dato_est", dato_est)

In [ ]:
# 2. OBTIENE DATOS A APLICAR PORCENTAJE
# data_fm_NAC y data_fm_ext: archivos SAS externos (.sas7bdat) no disponibles en el catálogo de conexiones
raise NotImplementedError("data_fm_NAC y data_fm_ext se leen de CARTERA_INV_NACIONAL.sas7bdat y CARTERA_INV_EXTERNA.sas7bdat (rutas SAS /sasdata/...), fuentes no disponibles en el catálogo de conexiones del proyecto")

In [ ]:
# concatena cartera nacional y externa (data_fm_ext sin la columna VALOR_REL_VAL)
data_fm_ext_drop = data_fm_ext.drop(columns=["VALOR_REL_VAL"])
data_fm = pd.concat([data_fm_nac, data_fm_ext_drop], ignore_index=True)

In [ ]:
# DATA_DEP_RUN: depósitos (DPC/DPL) por fondo, trimestres calendario
data_dep_run = (
    data_fm[data_fm["t_instcorto"].isin(["DPC", "DPL"]) & data_fm["mes"].isin([3, 6, 9, 12])]
    .merge(id_fm[["run_fondo", "Sector"]], on="run_fondo", how="left")
    .groupby(["año", "mes", "run_fondo", "Sector"], as_index=False)
    .agg(dato=("valor_mercado", "sum"))
)

# DATA_DEP: total depósitos por sector y trimestre
data_dep_run["trim"] = data_dep_run["mes"] / 3
data_dep = (
    data_dep_run.groupby(["año", "trim", "Sector"], as_index=False)
    .agg(dato=("dato", "sum"))
    .rename(columns={"Sector": "sector"})
)
_log("data_dep", data_dep)

In [ ]:
# 3. CALCULA DEPOSITOS A HOGARES
# tramo desde 2017: cruce por año, mes y run
dep_hh_2017 = dato_est[dato_est["AÑO"] >= 2017].merge(
    data_dep_run,
    left_on=["AÑO", "MES", "run_sdv"],
    right_on=["año", "mes", "run_fondo"],
    how="inner",
)
dep_hh_2017["trim"] = dep_hh_2017["mes"] / 3
dep_hh_2017 = (
    dep_hh_2017.assign(prod=dep_hh_2017["EST"] * dep_hh_2017["dato"])
    .groupby(["AÑO", "MES", "Sector"], as_index=False)
    .agg(dato=("prod", "sum"))
)
dep_hh_2017["trim"] = dep_hh_2017["MES"] / 3
dep_hh_2017 = dep_hh_2017.rename(columns={"AÑO": "año", "Sector": "sector"})[["año", "trim", "sector", "dato"]]

# tramo pre-2017: cruce solo por año y run (todos los meses)
dep_hh_pre2017_join = dato_est.merge(
    data_dep_run[data_dep_run["año"] < 2017],
    left_on=["AÑO", "run_sdv"],
    right_on=["año", "run_fondo"],
    how="inner",
)
dep_hh_pre2017_join["prod"] = dep_hh_pre2017_join["EST"] * dep_hh_pre2017_join["dato"]
dep_hh_2 = (
    dep_hh_pre2017_join.groupby(["año", "mes", "Sector"], as_index=False)
    .agg(dato=("prod", "sum"))
)
dep_hh_2["trim"] = dep_hh_2["mes"] / 3
dep_hh_2 = dep_hh_2.rename(columns={"Sector": "sector"})[["año", "trim", "sector", "dato"]]

dep_hh = pd.concat([dep_hh_2017, dep_hh_2], ignore_index=True)
_log("dep_hh", dep_hh)

In [ ]:
# % total para dep de hogares
dep_hh_fm = dep_hh.merge(data_dep, on=["año", "trim", "sector"], how="inner")
dep_hh_fm["dato"] = dep_hh_fm["dato_x"] / dep_hh_fm["dato_y"]
dep_hh_fm = dep_hh_fm[["año", "trim", "sector", "dato"]]

In [ ]:
# genera años faltantes: imputación determinística copiando 2006 trim=1
base_imputa = dep_hh_fm[(dep_hh_fm["año"] == 2006) & (dep_hh_fm["trim"] == 1)][["trim", "sector", "dato"]].copy()
imputa = base_imputa.assign(año=2005)[["año", "trim", "sector", "dato"]]
imputa_2 = base_imputa.assign(año=2004)[["año", "trim", "sector", "dato"]]
imputa_3 = base_imputa.assign(año=2003)[["año", "trim", "sector", "dato"]]
imputa = pd.concat([imputa, imputa_2, imputa_3], ignore_index=True)

imputa_a = imputa.copy()
imputa_a["trim"] = 2
imputa_b = imputa.copy()
imputa_b["trim"] = 3
imputa_c = imputa.copy()
imputa_c["trim"] = 4

# genera base serie completa
dep_hh_fm = pd.concat([dep_hh_fm, imputa, imputa_a, imputa_b, imputa_c], ignore_index=True)
_log("dep_hh_fm", dep_hh_fm)

In [ ]:
# para recalcular años de coyuntura o serie completa si es cierre de año
work_conn.execute(text("DROP TABLE IF EXISTS #dep_hh_fm_upload"))
dep_hh_fm.to_sql("#dep_hh_fm_upload", work_conn, if_exists="replace", index=False)

res = work_conn.execute(
    text("DELETE FROM TABLAS.dbo.DEP_HH_FM WHERE AÑO >= :anio"),
    {"anio": anio},
)
_log("DELETE TABLAS.dbo.DEP_HH_FM", res.rowcount)

dep_hh_fm_recalc = dep_hh_fm[dep_hh_fm["año"] >= anio]

In [ ]:
# APPEND a la tabla base (equivalente a PROC APPEND ... FORCE): columnas explícitas alineadas por nombre
cols_dep_hh_fm = ["año", "trim", "sector", "dato"]
dep_hh_fm_recalc[cols_dep_hh_fm].to_sql("DEP_HH_FM", engine, schema="dbo", if_exists="append", index=False)
_log("APPEND TABLAS.dbo.DEP_HH_FM", len(dep_hh_fm_recalc))

## Bonos_Ext

Precios implícitos de bonos externos (Valor Mercado/Valor Par) por sector y cuenta, para balance inicio y final de Gobierno, Empresas y Holdings

*confianza: medium · verificador: approve · SAS: PROC IMPORT + PROC SQL UPDATE/CREATE TABLE + DATA step SET (apilado) con recodificación de sectores*

In [ ]:
# ========= Bonos_Ext =========
# 1. IMPORTA DATA — bonos externos de la balanza de pagos (Excel), hoja DATA
# ruta relativa al workspace (M-001: elimina hardcode /samba/BCCH/...)
bonos_ext_path = Path("info_aux") / "bonos_ext_cdr18.xlsx"
bonos_ext = pd.read_excel(bonos_ext_path, sheet_name="DATA")


In [ ]:
# Recodificaciones de CNSI y C_CAGENTE previas al cálculo de precios
bonos_ext.loc[bonos_ext["CNSI"] == 5102, "CNSI"] = 51021
bonos_ext.loc[bonos_ext["CNSI"] == 322, "CNSI"] = 321
bonos_ext.loc[bonos_ext["C_CAGENTE"] != "6", "C_CAGENTE"] = "6"
# incorporado cierre 2025q2
bonos_ext.loc[bonos_ext["CNSI"] == 33, "CNSI"] = 36
_log("bonos_ext", bonos_ext)


In [ ]:
# Importa hoja DATA_EST y descarta filas sin año (año faltante en SAS == NaN en pandas)
bonos_ext_est_path = Path("info_aux") / "bonos_ext_cdr18.xlsx"
bonos_ext_est = pd.read_excel(bonos_ext_est_path, sheet_name="DATA_EST")
bonos_ext_est = bonos_ext_est[bonos_ext_est["año"].notna()]
_log("bonos_ext_est", bonos_ext_est)


In [ ]:
# CALCULA PRECIOS PARA GOBIERNO, EMPRESAS Y HOLDINGS
# CIERRE 2021: INCORPORA TMB BANCOS
# CIERRE 2022Q2: INCORPORA PRECIO DE BONOS EMITIDOS EN EL EXTERIOR DE AUXILIARES (36)
# Balance Final
bonos_ext_precio = (
    bonos_ext[
        bonos_ext["fuente"].isin(["Mercado Externo", "Mercado Externo (Recompras)"])
        & bonos_ext["CNSI"].isin([41, 37, 5101, 51021, 5102, 321, 36])
    ]
    .groupby(["Año", "Trim", "CNSI", "C_CAGENTE", "C_SCN", "C_CUENTA"], as_index=False)
    .agg(Valor_Mercado=("Valor_Mercado", "sum"), Valor_par=("Valor_par", "sum"))
)
bonos_ext_precio["Precio"] = bonos_ext_precio["Valor_Mercado"] / bonos_ext_precio["Valor_par"]
bonos_ext_precio = bonos_ext_precio.rename(columns={"CNSI": "Sector"})[
    ["Año", "Trim", "Sector", "C_CAGENTE", "C_SCN", "C_CUENTA", "Precio"]
]
_log("bonos_ext_precio", bonos_ext_precio)


In [ ]:
# Balance Final, para recompras en empresas
# SAS agrupa por C_CAGENTE ORIGINAL (no por la constante '54') y recién en el SELECT
# reetiqueta la columna como '54'; si un mismo Año/Trim/CNSI/C_SCN/C_CUENTA tiene
# más de un C_CAGENTE de origen, SAS produce filas separadas (luego todas '54').
bonos_ext_recompra = (
    bonos_ext[
        bonos_ext["fuente"].isin(["Mercado Externo (Recompras)"])
        & bonos_ext["CNSI"].isin([5101, 51021])
    ]
    .groupby(["Año", "Trim", "CNSI", "C_CAGENTE", "C_SCN", "C_CUENTA"], as_index=False)
    .agg(Valor_Mercado=("Valor_Mercado", "sum"), Valor_par=("Valor_par", "sum"))
)
bonos_ext_recompra["Precio"] = bonos_ext_recompra["Valor_Mercado"] / bonos_ext_recompra["Valor_par"]
bonos_ext_recompra["C_CAGENTE"] = "54"
bonos_ext_recompra = bonos_ext_recompra.rename(columns={"CNSI": "Sector"})[
    ["Año", "Trim", "Sector", "C_CAGENTE", "C_SCN", "C_CUENTA", "Precio"]
]
# Precio nulo (división por SUM(Valor_par)=0 o missing) se fuerza a 1
bonos_ext_recompra.loc[bonos_ext_recompra["Precio"].isna(), "Precio"] = 1
_log("bonos_ext_recompra", bonos_ext_recompra)


In [ ]:
# Apila recompras + balance final + estimaciones (DATA_EST)
bonos_ext_precio = pd.concat(
    [bonos_ext_recompra, bonos_ext_precio, bonos_ext_est], ignore_index=True
)
_log("bonos_ext_precio", bonos_ext_precio)


In [ ]:
# Balance Inicio: el trimestre 4 rueda al año siguiente, trim 1
bonos_ext_precio_2 = bonos_ext_precio.copy()
bonos_ext_precio_2["Año"] = np.where(
    bonos_ext_precio_2["Trim"] == 4, bonos_ext_precio_2["Año"] + 1, bonos_ext_precio_2["Año"]
)
bonos_ext_precio_2["Trim"] = np.where(
    bonos_ext_precio_2["Trim"] == 4, 1, bonos_ext_precio_2["Trim"] + 1
)
bonos_ext_precio_2["C_CUENTA"] = "Bce Inicio"
bonos_ext_precio_2 = bonos_ext_precio_2[
    ["Año", "Trim", "Sector", "C_CAGENTE", "C_SCN", "C_CUENTA", "Precio"]
]
_log("bonos_ext_precio_2", bonos_ext_precio_2)


In [ ]:
# Apila balance inicio + balance final y recodifica Sector 36 -> 36912
tablas_bonos_ext_precio = pd.concat(
    [bonos_ext_precio_2, bonos_ext_precio], ignore_index=True
)
tablas_bonos_ext_precio.loc[
    tablas_bonos_ext_precio["Sector"] == 36, "Sector"
] = 36912
_log("tablas_bonos_ext_precio", tablas_bonos_ext_precio)


In [ ]:
# Escritura idempotente en TABLAS.BONOS_EXT_PRECIO (SAS reemplazaba: DATA step sobre tabla existente)
with engine.begin() as conn:
    res = conn.execute(text("DELETE FROM TABLAS.dbo.BONOS_EXT_PRECIO"))
    _log("DELETE TABLAS.dbo.BONOS_EXT_PRECIO", res.rowcount)
tablas_bonos_ext_precio.to_sql(
    "BONOS_EXT_PRECIO", engine, schema="dbo", if_exists="append", index=False
)
